# ViViT 基线因子：端到端流程

本 Notebook 只负责逐步调用函数；数据、模型、训练和校验逻辑均位于 `vivit_factor` 包中。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from vivit_factor.data import (
    PipelinePaths, build_feature_store, build_history_indices, build_labels,
    build_rolling_splits, build_universe, validate_inputs,
)
from vivit_factor.training import run_parallel_training, smoke_test, validate_outputs

paths = PipelinePaths.default()
OVERWRITE_DATA = False
DEVICE_IDS = (0, 1, 2, 3)
TASKS_PER_GPU = 1
WORKERS_PER_TASK = 8

## 1. 校验 Level2 图片与 Wind 行情输入

In [ ]:
validate_inputs(paths)

## 2. 构建并检查指定口径的股票池

In [ ]:
universe_manifest = build_universe(paths, overwrite=OVERWRITE_DATA)
universe_manifest

## 3. 构建复权 VWAP 标签并检查 MAD 去极值与标准化结果

In [ ]:
label_manifest = build_labels(paths, overwrite=OVERWRITE_DATA)
label_sample = pd.read_parquet(paths.labels_dir / 'labels.2018.parquet')
display(label_sample.head())
display(label_sample.groupby('trade_date')['target_zscore'].agg(['count', 'mean', lambda x: x.std(ddof=0)]).head())
label_manifest

## 4. 构建标准化特征与样本索引

In [ ]:
feature_manifest = build_feature_store(paths, overwrite=OVERWRITE_DATA)
sample_index = pd.read_parquet(paths.sample_index_path)
display(sample_index.head())
feature_manifest

## 5. 构建 20 日精确历史窗口与滚动样本外切分

In [ ]:
build_history_indices(sample_index, paths.history_path)
split_summary = build_rolling_splits(paths)
display(split_summary)

## 6. 执行单批次前向与反向冒烟测试

In [ ]:
smoke_test(paths, forecast_year=2018, seed=42)

## 7. 在多张 GPU 上运行独立的年度 × 随机种子任务

这是耗时最长的步骤；签名一致的 checkpoint 会自动恢复。

In [ ]:
training_summary = run_parallel_training(
    paths, device_ids=DEVICE_IDS, tasks_per_gpu=TASKS_PER_GPU,
    workers=WORKERS_PER_TASK, overwrite=False,
)
display(training_summary)

## 8. 校验各年度样本外因子

In [ ]:
validation = validate_outputs(paths)
display(validation)
print(f'因子目录：{paths.factors_dir}')